# Documentação — Projeto Celulares SP

Este documento registra, de forma acessível, todo o caminho percorrido no projeto de análise de furtos e roubos de celulares (dados da SSP-SP): como os dados chegaram, como foram organizados, quais problemas de qualidade encontramos e como os resolvemos.

**Última atualização:** 26/10/2026

## 1. A arquitetura em camadas (Medallion)

Organizamos o projeto no padrão **Medalhão**, que divide o tratamento dos dados em camadas, como uma esteira:

| Camada | Nome | O que significa |
| --- | --- | --- |
| 🥉 **Bronze** | Dados brutos | Os dados exatamente como chegaram da fonte, sem nenhum ajuste. Serve como "cópia de segurança" — nunca alteramos o que está aqui. |
| 🥈 **Silver** | Dados tratados | Os dados limpos, padronizados e corrigidos. É aqui que acontecem as correções e imputações descritas neste documento. |
| 🥇 **Gold** | Dados prontos para análise | Camada final, com indicadores e tabelas otimizadas para dashboards e modelos. **Ainda será construída.** |

**Fluxo atual (pipeline completo no Visual Data Prep):**

```text
Arquivos Parquet (SSP-SP)
        │
        ▼  ingestão
[CAMADA BRONZE]  celulares_bronze
        │
        ▼  Visual Data Prep
    Transformacao_Silver (limpeza e tipagem)
        │
        ▼  nó SQL (SELECT)
    Imputa_DESCR_PERIODO (hora → período)
        │
        ▼  nó SQL (SELECT)
    Imputação por moda estratificada
        │
        ▼  operador de transformação
    Ordena_colunas (reordena 58 colunas)
        │
        ▼  Output
[CAMADA SILVER]  silver_celulares_sp (final, com imputações)
        │
        ▼  (próxima etapa)
[CAMADA GOLD]
```

> ✅ **Atualização (22/10):** as imputações agora são feitas **dentro do pipeline** via nós SQL — não é mais necessário executar o notebook separadamente.

## 2. Ingestão — Camada Bronze 🥉

**Fonte:** Boletins de ocorrência de furto/roubo de celular da Secretaria de Segurança Pública de São Paulo (SSP-SP), recebidos em arquivos **XLSX**.

* **Localização dos arquivos:** volume `/Volumes/workspace/treinamentos/projeto_celulares_sp/`
* **Arquivos originais:** 2 arquivos **XLSX** (referentes a 2025 e 2026, com pouquíssimos registros de 2022–2024), convertidos para **Parquet** durante a ingestão
* **Volume de armazenamento:** `/Volumes/workspace/treinamentos/projeto_celulares_sp/` (arquivos Parquet já convertidos)
* **Total de registros:** 1.222.380
* **Abrangência:** 628 municípios paulistas

A tabela **`workspace.treinamentos.celulares_bronze`** guarda esses dados a partir dos Parquet já convertidos. Nesse estado, os dados tinham características que impediam qualquer análise direta:

* **Todas as colunas eram texto** — inclusive datas e horas (ex.: a data de ocorrência era uma frase como `2025-01-01`, não uma data de verdade);
* **Dados ausentes apareciam como o texto `"nan"`** — ou seja, quando a informação não existia, o arquivo trazia a palavra "nan" escrita, e não um campo vazio de verdade;
* Alguns campos tinham formatação irregular (pontos no CEP, textos em caixas variadas, etc.).

> 💡 A camada Bronze existe justamente para preservar esse "estado original". Qualquer correção acontece nas camadas seguintes, mantendo a rastreabilidade.

## 3. Camada Silver — Limpeza com o Visual Data Prep 🥈

A primeira grande transformação foi feita na ferramenta **Visual Data Prep** (pipeline visual), no operador **"Transformacao_Silver"** (antigo "Limpeza_Silver"). Ele lê a Bronze e faz a limpeza inicial. O pipeline completo agora tem **8 nós** encadeados, terminando na tabela **`workspace.treinamentos.silver_celulares_sp`** — que já nasce limpa, com **610.366 registros** (limpezas adicionais e imputação de coordenadas incorporadas em 26/10, seção 6).

### O que o Limpeza_Silver faz

1. **Limpa valores ausentes:** os textos `"nan"` viram campos vazios de verdade (nulos), para que o banco entenda que ali "não tem informação".
2. **Converte tipos de dados:** datas como `DATA_OCORRENCIA_BO`, `DATA_COMUNICACAO_BO` e `DATAHORA_*` deixaram de ser texto e passaram a ser datas de verdade; números como `ID_DELEGACIA`, `ANO_BO` e `LATITUDE`/`LONGITUDE` passaram a ser numéricos.
3. **Padroniza textos:** os campos descritivos ficam em caixa alta, facilitando agrupamentos e filtros.
4. **Cria a datahora completa:** a coluna `DATAHORA_OCORRENCIA` combina a data com a hora da ocorrência em um único campo.
5. **Limpa o CEP:** remove os pontos do campo `CEP` (ex.: `01234.567` vira `01234567`).
   * ⚠️ *Detalhe técnico:* no editor de Regex, o padrão precisa ser `\.` (ponto escapado). Um ponto solto (`.`) em expressão regular significa "qualquer caractere" e apagaria o CEP inteiro.

### Problemas de qualidade que motivaram as etapas seguintes

Durante a exploração dos dados, encontramos dois pontos importantes na coluna de período do crime:

| Campo | Situação |
| --- | --- |
| `DESCR_PERIODO` | 332.327 registros (54,4%) estavam **vazios** |
| `DESCR_PERIODO` | 5.939 registros marcados como **`EM HORA INCERTA`** — e nesses, a hora também não existe |
| `HORA_OCORRENCIA` | 278.863 registros (45,6%) **não têm hora na origem** — isso é característica dos dados, não erro de processamento |

Essas lacunas distorceriam qualquer análise por período do dia — por isso as imputações, agora incorporadas ao próprio pipeline (próxima seção).

## 4. Imputações — Nós SQL no Visual Data Prep

As imputações são o preenchimento inteligente de lacunas. Originalmente foram feitas no notebook **Silver Celulares SP** com `UPDATE`/`MERGE`. Agora foram **incorporadas ao pipeline Visual Data Prep como nós SQL com `SELECT`**, sempre respeitando a regra: **nunca sobrescrever informação que já existe**.

### Por que SELECT e não operações DML no Visual Data Prep?

O Visual Data Prep funciona como um **fluxo de dados**: cada nó recebe linhas do nó anterior e devolve linhas para o próximo. Operações DML (`UPDATE`, `MERGE`, `DELETE`) **não retornam linhas** — elas modificam uma tabela existente e devolvem apenas um contador de linhas afetadas. Por isso:

* O nó SQL do Data Prep **exige um `SELECT`** que retorne colunas e linhas;
* Mesmo que um `UPDATE` rodasse antes do nó de saída, o Output faz `CREATE OR REPLACE TABLE`, **recriando a tabela do zero** e apagando qualquer alteração feita por DML;
* A solução é usar `SELECT * EXCEPT (coluna)` com `CASE WHEN` para transformar a coluna **no fluxo de dados**, sem modificar a tabela por fora.

### Etapa 1 — Preencher o período usando a hora (nó SQL "Imputa_DESCR_PERIODO")

**Para quem:** os 332.327 registros com `DESCR_PERIODO` vazio **mas com hora preenchida**.

**Como:** o nó SQL usa `HOUR(DATAHORA_OCORRENCIA)` para determinar a faixa do dia. A coluna `DATAHORA_OCORRENCIA` foi ajustada para retornar `NULL` quando a hora não existe (evitando classificação errada como madrugada):

| Faixa | Período atribuído |
| --- | --- |
| 00:00 às 06:00 | DE MADRUGADA |
| 06:00 às 12:00 | PELA MANHÃ |
| 12:00 às 18:00 | A TARDE |
| 18:00 às 24:00 | A NOITE |

**Resultado:** 332.327 registros preenchidos — e a distribuição conferiu exatamente com a expectativa (38,0% à noite, 25,2% à tarde, 20,3% pela manhã, 16,6% de madrugada).

### Etapa 2 — Preencher o "EM HORA INCERTA" pela moda estratificada (nó SQL "Imputação por moda estratificada")

**Para quem:** os 5.939 registros marcados como `EM HORA INCERTA` — que **não têm hora na origem**, impossibilitando o método da Etapa 1.

**Como:** procuramos registros "parecidos" que têm o período conhecido e copiamos o período **mais comum** (a *moda*) entre eles. A semelhança é testada em cascata, do mais específico ao mais geral:

1. **Mesmo local + mesmo tipo de crime + mesmo dia da semana** → usado para 5.860 registros (98,7%)
2. Se não houver: **mesmo local + mesmo tipo de crime** → 62 registros
3. Se não houver: **mesmo tipo de crime** → 14 registros
4. Se nada disso existir: **o período mais comum da base inteira** (A NOITE) → 3 registros

**Resultado:** todos os 5.939 registros imputados. A implementação usa CTEs encadeadas (`base` → `moda_n1..n4`) com `QUALIFY ROW_NUMBER() = 1` e `COALESCE` para aplicar a cascata de fallback em um único `SELECT`.

### Como ficou a tabela ao final

| Período | Registros | % |
| --- | --- | --- |
| A NOITE | 209.616 | 34,3% |
| A TARDE | 165.276 | 27,0% |
| PELA MANHÃ | 135.198 | 22,1% |
| DE MADRUGADA | 101.100 | 16,5% |
| *(nulos ou incertos)* | **0** | — |

> ℹ️ **Nota:** a coluna `HORA_OCORRENCIA` continua com 45,6% de campos vazios. Isso é **correto** — esses horários nunca existiram na fonte. O que fizemos foi garantir que **nenhum registro fique sem o seu período do dia**.

### Operador "Ordena_colunas" — reordenação das colunas

Como o `SELECT * EXCEPT (DESCR_PERIODO)` reposiciona a coluna `DESCR_PERIODO` para o final da tabela, adicionamos um operador de transformação **"Ordena_colunas"** após o nó sql_4 para restaurar a ordem original das 58 colunas — com `DESCR_PERIODO` de volta à posição 19 (entre `DATAHORA_IMPRESSAO_BO` e `AUTORIA_BO`).

> ⚠️ *Detalhe técnico:* `SELECT * REPLACE (...)` não é suportado pelo parser do Spark nesta versão, o que impedia apenas substituir o valor mantendo a posição. A solução foi usar um operador dedicado de reordenação.

### Pipeline completo (5 nós)

| Ordem | Nó | Tipo | Função |
| --- | --- | --- | --- |
| 1 | **Transformacao_Silver** | Transformação | Limpeza, tipagem e padronização (limpa "nan", converte datas, normaliza textos, cria `DATAHORA_OCORRENCIA`, limpa CEP) |
| 2 | **Imputa_DESCR_PERIODO** (sql_3) | SQL (`SELECT`) | Imputação do período pela hora — `CASE WHEN HOUR(DATAHORA_OCORRENCIA) ...` |
| 3 | **Imputação por moda estratificada** (sql_4) | SQL (`SELECT`) | Imputação dos `EM HORA INCERTA` via cascata de modas |
| 4 | **Ordena_colunas** | Transformação | Reordena as 58 colunas para a ordem original |
| 5 | **Output Silver_Celulares_SP** | Saída | `CREATE OR REPLACE TABLE workspace.treinamentos.silver_celulares_sp` |


## 6. Limpeza adicional e imputação de coordenadas (26/10/2026) 🧹📍

Nova rodada de qualidade de dados, executada originalmente no notebook **Silver Celulares SP** e depois **incorporada ao próprio pipeline Visual Data Prep** (seção 6.5). Tudo validado: cada etapa teve a contagem conferida antes e depois da operação, e o pipeline reproduz exatamente o mesmo resultado.

### 6.1 Registros inválidos removidos (457 no total)

| Problema encontrado | Exemplos | Removidos | Critério usado |
| --- | --- | --- | --- |
| **Coordenadas multiplicadas por 10.000** — valores impossíveis (SP tem lat ≈ -23,5 e long ≈ -46,6) | -235.091/-466.252 (Canindé, 55 registros); -2.363.499/-4.664.112 (Jabaquara, 23) | **102** | `LATITUDE <= -100 AND LONGITUDE <= -100` |
| **Ocorrências anteriores a 2020**, fora do recorte temporal do projeto (mais antiga: 25/11/2005) | concentração em 2015 (172 registros) e 2016 (59) | **296** | `DATA_OCORRENCIA_BO < '2020-01-01'` |
| **Quantidades extremas** de celulares em um único BO | 1.453 celulares APPLE (roubo), 999 APPLE (furto), 400 XIAOMI (Guarulhos) | **59** | `QUANTIDADE_OBJETO > 100` |

### 6.2 Duplicatas exatas (367 linhas)

Encontramos **367 linhas 100% idênticas em todas as 58 colunas** (ex.: o BO DO2032/2026 aparecia em triplicata). Como cópias idênticas não podem ser distinguidas por um `DELETE` (qualquer critério atingiria todas), regravamos a tabela mantendo uma única cópia de cada linha (`SELECT DISTINCT`). O histórico da tabela (time travel) foi preservado.

### 6.3 Imputação de coordenadas (LATITUDE e LONGITUDE)

**Situação inicial:** 35.209 registros com coordenadas "zero" (0,0 — valor de preenchimento, não um local real) e 73.217 sem coordenada. Os campos `BAIRRO`, `CIDADE` e `LOGRADOURO` estavam preenchidos nesses registros, e 501.940 registros já tinham coordenadas válidas para servir de referência.

**Estratégia hierárquica** (nunca sobrescrever coordenada válida existente):

1. **Nível 1:** média das coordenadas dos registros do mesmo **BAIRRO + CIDADE**;
2. **Nível 2:** sem bairro compatível → média da **CIDADE**;
3. **Nível 3:** sem referência → **NULL** (melhor "vazio" do que local errado).

**Resultado:**

| Destino | Registros | Detalhe |
| --- | --- | --- |
| Coordenada original preservada | 501.940 | válidas na fonte, intactas |
| Imputada por BAIRRO + CIDADE | 93.708 | 27.794 zeros + 65.914 nulas |
| Imputada por CIDADE | 14.615 | 7.334 zeros + 7.281 nulas |
| Sem referência → NULL | 103 | 81 zeros + 22 nulas |

**Verificações:** nenhuma coordenada original válida foi alterada, nenhuma imputada caiu fora da faixa de São Paulo, e as médias gerais conferem com a tabela de auditoria (diferença apenas de precisão numérica).

**Tabela de auditoria:** `workspace.treinamentos.silver_celulares_sp_imputado` guarda, para cada registro, a coordenada original, a final, a flag de origem (`ORIGINAL` / `IMPUTADO_BAIRRO` / `IMPUTADO_CIDADE` / `NAO_IMPUTADO`), a quantidade de referências usadas e o desvio-padrão do grupo — permitindo medir a confiabilidade de cada coordenada imputada.

> ⚠️ *Detalhe técnico:* as referências precisam ficar **congeladas** durante a imputação em dois estágios. Usamos como fonte apenas os registros com coordenada original válida — se o segundo estágio recalculasse as médias incluindo linhas já imputadas pelo primeiro, os valores deslocariam (problema detectado e corrigido durante a implementação, com reversão via time travel).

### 6.4 Estado final da Silver (26/10/2026)

| Métrica | Valor |
| --- | --- |
| Registros | 610.366 |
| Com coordenadas válidas | 610.263 (99,98%) |
| Sem coordenada (NULL) | 103 |
| Placeholders (0,0) e duplicatas exatas | 0 |
| Recorte temporal das ocorrências | 21/01/2020 a 31/08/2026 |

### 6.5 ✅ Incorporado ao pipeline (26/10/2026)

Todas as etapas desta seção viraram **nós SQL do pipeline Visual Data Prep**, logo após os nós de imputação de período:

| Nó | O que faz |
| --- | --- |
| **Limpeza_e_dedup** | `SELECT DISTINCT *` com filtros `IS NOT TRUE` que removem os 457 registros inválidos e as 367 duplicatas — tudo em um único nó de fluxo, sem DELETE |
| **Imputacao_coordenadas** | imputação hierárquica completa (bairro → cidade → NULL) via CTEs de referência e `CASE WHEN`, gravando direto nas colunas LATITUDE/LONGITUDE — sem tabela auxiliar |

> ✅ **Resultado validado:** simulando os nós sobre o estado anterior à limpeza, o pipeline reproduz **exatamente, linha por linha** (comparação `EXCEPT` com 0 diferenças nas duas direções) a Silver validada: 610.366 registros, 610.263 com coordenadas válidas e 103 sem coordenada. O nó `Ordena_colunas` continua devolvendo LATITUDE/LONGITUDE às posições 39/40.
>
> 💡 O notebook **Silver Celulares SP** permanece como backup: suas 6 células são a versão DML (DELETE/MERGE/UPDATE) do mesmo processo.

## 5. ✅ Imputações incorporadas ao pipeline (22/10/2026)

**Atualização importante:** as imputações agora são **parte do pipeline Visual Data Prep** — executadas automaticamente em cada rodada via nós SQL com `SELECT`. Não é mais necessário executar o notebook separadamente após cada execução do pipeline.

**Histórico:** as imputações foram originalmente implementadas no notebook **Silver Celulares SP** usando `UPDATE` e `MERGE` (operações DML). Como o Visual Data Prep não aceita DML em seus nós SQL (cada nó deve retornar linhas via `SELECT`, e o Output recria a tabela com `CREATE OR REPLACE`), as imputações foram **reescritas como `SELECT`** e incorporadas ao fluxo.

> 💡 O notebook permanece como **backup e validação** (versão DML das mesmas transformações, incluindo as de coordenadas da seção 6) — suas células podem ser executadas manualmente caso seja necessário reaplicar tudo sem rodar o pipeline completo.

## 7. Próximos passos

* 🥇 **Camada Gold:** criar tabelas analíticas (por município, por período, por tipo de crime) para dashboards;
* **Decodificar** os códigos de autoria (`AUTORIA_BO`: D/S/C) e de intolerância (`FLAG_INTOLERANCIA`);
* **Investigar** os anos 2022–2024 (apenas 126 registros — possível problema na fonte);
* **Derivar colunas para análise/ML:** dia da semana, flag de fim de semana, faixa horária granular;
* ✅ **Feito (26/10):** limpezas e imputação de coordenadas incorporadas ao pipeline (nós Limpeza_e_dedup e Imputacao_coordenadas) — a Silver já nasce limpa, sem depender do notebook.

---
*Documentação atualizada em 26/10/2026 — Projeto Celulares SP (SSP-SP · Arquitetura Medalhão · Databricks)*